In [ ]:
import os
import requests
import pandas as pd
import logging
from dotenv import load_dotenv
from datetime import datetime, timedelta
from time import sleep, time as now

# === LOGGING ===
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# === CONFIG ===
force_clean_start = True  # ✅ set True for full reset
output_dir = r"C:\Android Mobile App\Step 1_URL_Search_Kotlin"
output_csv_raw = os.path.join(output_dir, "github_android_search_results_raw.csv")
output_csv_filtered = os.path.join(output_dir, "github_android_search_results_filtered.csv")
output_ranges_csv = os.path.join(output_dir, "final_ranges_used.csv")

os.makedirs(output_dir, exist_ok=True)

# === AUTH ===
load_dotenv("All_Tokens.env")
tokens = [v for k, v in os.environ.items() if k.startswith("GITHUB_TOKEN_") and v]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

token_index = 0
HEADERS = {
    "Authorization": f"token {tokens[token_index]}",
    "Accept": "application/vnd.github+json"
}

# === DATE RANGE ===
start_date = datetime.strptime("2008-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2024-12-31", "%Y-%m-%d")

# === WINDOW SETTINGS ===
initial_window_hours = 15 * 24  # 10 days
min_window_hours = 1
max_window_hours = 90 * 24
MAX_RESULTS_PER_QUERY = 1000
TARGET_FILL_RATIO = 0.25

# === FUNCTIONS ===
def rotate_token():
    global token_index, HEADERS
    token_index = (token_index + 1) % len(tokens)
    HEADERS = {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github+json"
    }
    logger.warning("\n⚠️" + "="*40 + f"🔑 Token Rotated → Using token #{token_index + 1} / {len(tokens)}\n" + "="*40)


def check_rate_limit():
    r = requests.get("https://api.github.com/rate_limit", headers=HEADERS)
    if r.status_code != 200:
        logger.warning("⚠️  Could not check rate limit.")
        return
    data = r.json()
    remaining = data['resources']['search']['remaining']
    reset_epoch = data['resources']['search']['reset']
    reset_in = max(0, reset_epoch - now())
    logger.info(f"🔎 Search API remaining: {remaining} requests | resets in {reset_in/60:.1f} min")
    if remaining < 5:
        logger.warning(f"⏳ Low quota. Sleeping for {reset_in/60:.1f} min.")
        sleep(reset_in + 5)
        check_rate_limit()


def check_count(query):
    check_rate_limit()
    params = {"q": query, "per_page": 1}
    r = requests.get("https://api.github.com/search/repositories", headers=HEADERS, params=params)
    if r.status_code == 403:
        rotate_token()
        return check_count(query)
    if r.status_code != 200:
        logger.error(f"❌ Count check error: {r.status_code} — {r.text}")
        return -1
    return r.json().get("total_count", 0)


def fetch_items(query):
    all_items = []
    per_page = 100
    max_pages = 10
    for page in range(1, max_pages + 1):
        check_rate_limit()
        params = {"q": query, "per_page": per_page, "page": page}
        r = requests.get("https://api.github.com/search/repositories", headers=HEADERS, params=params)
        if r.status_code == 403:
            rotate_token()
            return fetch_items(query)
        if r.status_code != 200:
            logger.error(f"❌ Fetch error: {r.status_code} — {r.text}")
            break
        data = r.json().get("items", [])
        if not data:
            break
        all_items.extend(data)
        sleep(2)
    return all_items


# === BASE QUERY SETUP ===
base_queries = [
    "stars:>0 language:Kotlin fork:true archived:false",
    "stars:>0 language:Kotlin fork:false archived:true",
    "stars:>0 language:Kotlin fork:false archived:false"
]

for base_query_prefix in base_queries:

    # === SMART RESUME ===
    if os.path.exists(output_ranges_csv):
        df_ranges = pd.read_csv(output_ranges_csv)
        if not df_ranges.empty:
            last_end = df_ranges['end_date'].iloc[-1]
            current_start = datetime.fromisoformat(last_end) + timedelta(seconds=1)
            logger.info(f"⏮️ Resuming from {current_start}")
        else:
            current_start = start_date
    else:
        current_start = start_date

    if force_clean_start:
        for file in [output_csv_raw, output_csv_filtered, output_ranges_csv]:
            if os.path.exists(file):
                os.remove(file)
                logger.info(f"🧹 Removed old file: {file}")
        current_start = start_date

    # === INIT MEMORY ===
    all_results = []
    final_ranges = []

    if os.path.exists(output_csv_raw):
        all_results = pd.read_csv(output_csv_raw).to_dict('records')
        logger.info(f"♻️ Loaded existing raw results: {len(all_results)}")
    if os.path.exists(output_ranges_csv):
        final_ranges = pd.read_csv(output_ranges_csv).to_dict('records')
        logger.info(f"♻️ Loaded existing ranges: {len(final_ranges)}")

    # === SMART LOOP WITH AUTO SHRINK/EXPAND ===
    current_window_hours = initial_window_hours

    while current_start < end_date:
        window_hours = current_window_hours

        while True:
            current_end = current_start + timedelta(hours=window_hours)
            if current_end > end_date:
                current_end = end_date

            date_range = f"created:{current_start.isoformat()}..{current_end.isoformat()}"
            base_query = f"{base_query_prefix} {date_range}"

            total_count = check_count(base_query)
            logger.info(f"⏳ Checking: {base_query} → {total_count} repos")

            if total_count >= MAX_RESULTS_PER_QUERY and window_hours > min_window_hours:
                window_hours = max(window_hours // 2, min_window_hours)
                logger.info(f"⚠️ Too many results ({total_count}). Shrinking window to {window_hours} hours and retrying.")
                continue

            elif total_count < MAX_RESULTS_PER_QUERY * TARGET_FILL_RATIO and window_hours * 2 <= max_window_hours:
                window_hours = min(window_hours * 2, max_window_hours)
                logger.info(f"✅ Few results ({total_count}). Will expand window to {window_hours} hours next time.")

            items = fetch_items(base_query)
            items = [item for item in items if item.get('stargazers_count', 0) > 0]

            for item in items:
                item['search_qualifier'] = base_query
                item['repo_stars'] = item.get('stargazers_count', 0)
                item['match_type'] = "Kotlin"

            all_results.extend(items)
            final_ranges.append({
                "start_date": current_start.isoformat(),
                "end_date": current_end.isoformat(),
                "result_count": total_count,
                "window_hours": window_hours,
                "match_type": "Kotlin"
            })

            df_tmp = pd.json_normalize(all_results)
            df_tmp.to_csv(output_csv_raw, index=False)
            df_ranges_tmp = pd.DataFrame(final_ranges)
            df_ranges_tmp.to_csv(output_ranges_csv, index=False)
            logger.info(f"💾 Progress saved: {len(all_results)} repos, {len(final_ranges)} ranges so far")

            sleep(3)
            break

        current_start = current_end + timedelta(seconds=1)
        current_window_hours = window_hours

# === FINAL FILTER ===
df = pd.json_normalize(all_results)
keep_fields = [
    'language', 'search_qualifier', 'match_type', 'name', 'full_name', 'private', 'html_url', 'url',
    'clone_url', 'visibility', 'owner.login', 'size', 'stargazers_count',
    'watchers_count', 'forks', 'open_issues', 'default_branch',
    'open_issues_count', 'repo_stars', 'topics', 'description', 'fork', 'archived'
]

for field in keep_fields:
    if field not in df.columns:
        df[field] = None

df = df[keep_fields]
df.to_csv(output_csv_filtered, index=False)
logger.info(f"✅ Filtered fields saved to: {output_csv_filtered}")


[INFO] ⏮️ Resuming from 2019-08-16 00:01:45
C:\Users\gilla\AppData\Local\Temp\ipykernel_15164\463342136.py:145: DtypeWarning: Columns (64) have mixed types. Specify dtype option on import or set low_memory=False.
  all_results = pd.read_csv(output_csv_raw).to_dict('records')
[INFO] ♻️ Loaded existing raw results: 38287
[INFO] ♻️ Loaded existing ranges: 105
[INFO] 🔎 Search API remaining: 30 requests | resets in 1.0 min
[INFO] ⏳ Checking: stars:>0 language:Kotlin fork:false archived:false created:2019-08-16T00:01:45..2019-08-31T00:01:45 → 800 repos
[INFO] 🔎 Search API remaining: 29 requests | resets in 1.0 min
[INFO] 🔎 Search API remaining: 28 requests | resets in 0.9 min
[INFO] 🔎 Search API remaining: 27 requests | resets in 0.8 min
[INFO] 🔎 Search API remaining: 26 requests | resets in 0.8 min
[INFO] 🔎 Search API remaining: 25 requests | resets in 0.7 min
[INFO] 🔎 Search API remaining: 24 requests | resets in 0.6 min
[INFO] 🔎 Search API remaining: 23 requests | resets in 0.5 min
[INFO]